# v5: 2段階アプローチ（FastICA → GIN）合成実験

**目的**: AE + Jacobian L1（論文の方法、46回全滅）の代替として、2段階アプローチで潜在変数の分離（もつれほどき）を検証する。

**手法**:
```
X → FastICA → Z_ica → GIN(NLL) → Z_hat
     ↑ 線形成分を除去    ↑ 非線形残差のみ学習
```

**先行結果（ローカル検証済み）**: dim=16, 非線形混合で MCC=0.754 (best), 0.728 (mean)

**このノートブックの目標**: dim=32, 64, 128 での高次元テスト（GPU使用）

In [ ]:
# === Cell 1: セットアップ ===
import os, sys

# Colab環境判定
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # リポジトリをclone
    if not os.path.exists('thoughtcomm'):
        !git clone https://github.com/AUMEZAK/thoughtcomm.git
    %cd thoughtcomm
    !pip install -e . -q
    !pip install FrEIA -q
else:
    # ローカル実行
    if os.path.basename(os.getcwd()) != 'thoughtcomm':
        os.chdir(os.path.join(os.path.dirname(os.getcwd()), 'thoughtcomm'))
    sys.path.insert(0, '.')

import torch
import torch.nn as nn
import numpy as np
import json
import time
from scipy.optimize import linear_sum_assignment
from scipy.stats import pearsonr
from sklearn.decomposition import FastICA

import FrEIA.framework as Ff
import FrEIA.modules as Fm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# === Cell 2: ユーティリティ関数 ===

def compute_mcc(z_hat_np, z_true_np):
    """Mean Correlation Coefficient (最適置換マッチング)"""
    dim = z_true_np.shape[1]
    corr = np.zeros((dim, dim))
    for i in range(dim):
        for j in range(dim):
            corr[i, j] = abs(pearsonr(z_hat_np[:, i], z_true_np[:, j])[0])
    row, col = linear_sum_assignment(-corr)
    mcc = corr[row, col].mean()
    perm = np.argsort(col)  # 逆置換
    return mcc, perm


def compute_r2_matrix(z_hat_np, z_true_np, group_indices, perm):
    """R² matrix between estimated and true latent groups."""
    from sklearn.linear_model import LinearRegression
    z_hat_perm = z_hat_np[:, perm]
    groups = list(group_indices.keys())
    n_groups = len(groups)
    r2 = np.zeros((n_groups, n_groups))
    for i, gi in enumerate(groups):
        for j, gj in enumerate(groups):
            si, ei = group_indices[gi]
            sj, ej = group_indices[gj]
            reg = LinearRegression().fit(z_hat_perm[:, si:ei], z_true_np[:, sj:ej])
            r2[i, j] = max(0, reg.score(z_hat_perm[:, si:ei], z_true_np[:, sj:ej]))
    return r2, groups


def make_nonlinear_data(dim, n_samples, num_mix_layers=3, seed=42):
    """非線形混合の合成データ生成（Laplace(0,1)ソース）"""
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # 潜在変数: Laplace(0, 1)
    Z = torch.distributions.Laplace(0, 1).sample((n_samples, dim))
    
    # 非線形混合: multi-layer MLP with orthogonal init
    layers = []
    for i in range(num_mix_layers):
        layers.append(nn.Linear(dim, dim))
        if i < num_mix_layers - 1:
            layers.append(nn.LeakyReLU(0.2))
    mix_net = nn.Sequential(*layers)
    for m in mix_net.modules():
        if isinstance(m, nn.Linear):
            nn.init.orthogonal_(m.weight)
            nn.init.zeros_(m.bias)
    mix_net.eval()
    with torch.no_grad():
        X = mix_net(Z)
    
    # グループインデックス（2エージェント設定）
    n_shared = max(1, dim // 3)
    n_private_a = (dim - n_shared) // 2
    n_private_b = dim - n_shared - n_private_a
    group_indices = {
        'Z_A_private': (0, n_private_a),
        'Z_shared': (n_private_a, n_private_a + n_shared),
        'Z_B_private': (n_private_a + n_shared, dim),
    }
    
    return X.numpy(), Z.numpy(), group_indices


def make_linear_data(dim, n_samples, seed=42):
    """線形混合の合成データ生成（Laplace(0,1)ソース）"""
    torch.manual_seed(seed)
    np.random.seed(seed)
    Z = torch.distributions.Laplace(0, 1).sample((n_samples, dim))
    W = torch.randn(dim, dim)
    Q, _ = torch.linalg.qr(W)
    with torch.no_grad():
        X = Z @ Q.T
    n_shared = max(1, dim // 3)
    n_private_a = (dim - n_shared) // 2
    group_indices = {
        'Z_A_private': (0, n_private_a),
        'Z_shared': (n_private_a, n_private_a + n_shared),
        'Z_B_private': (n_private_a + n_shared, dim),
    }
    return X.numpy(), Z.numpy(), group_indices


print('Utilities loaded.')

In [ ]:
# === Cell 3: 2段階パイプライン ===

def stage1_ica(X_np, dim, random_state=42):
    """Stage 1: FastICA で線形成分を除去"""
    ica = FastICA(n_components=dim, max_iter=2000, tol=1e-4, random_state=random_state)
    Z_ica = ica.fit_transform(X_np)
    return Z_ica, ica


def stage2_gin(Z_ica_np, Z_true_np, dim, n_coupling=8, subnet_hidden=None,
               n_epochs=500, lr=5e-4, weight_decay=0.01, batch_size=256,
               seed=42, verbose=True):
    """Stage 2: GIN (NLL with Laplace prior) で非線形残差を学習"""
    if subnet_hidden is None:
        subnet_hidden = max(dim, 10)
    
    torch.manual_seed(seed)
    
    def subnet_fc(dims_in, dims_out):
        return nn.Sequential(
            nn.Linear(dims_in, subnet_hidden), nn.ReLU(),
            nn.Linear(subnet_hidden, subnet_hidden), nn.ReLU(),
            nn.Linear(subnet_hidden, dims_out),
        )
    
    inn = Ff.SequenceINN(dim).to(device)
    for _ in range(n_coupling):
        inn.append(Fm.GINCouplingBlock, subnet_constructor=subnet_fc, clamp=2.0)
        inn.append(Fm.PermuteRandom)
    
    # 正規化
    Z_ica = torch.tensor(Z_ica_np, dtype=torch.float32)
    z_mean = Z_ica.mean(0)
    z_std = Z_ica.std(0).clamp(min=1e-8)
    Z_norm = ((Z_ica - z_mean) / z_std).to(device)
    n_samples = Z_norm.shape[0]
    
    optimizer = torch.optim.Adam(inn.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    
    best_mcc = 0
    loss_history = []
    
    for epoch in range(n_epochs):
        inn.train()
        perm = torch.randperm(n_samples)
        epoch_nll = 0
        n_batch = 0
        
        for i in range(0, n_samples, batch_size):
            idx = perm[i:i+batch_size].to(device)
            z_batch = Z_norm[idx]
            z_hat, _ = inn(z_batch)
            nll = (z_hat.abs() + np.log(2)).sum(dim=1).mean()
            
            optimizer.zero_grad()
            nll.backward()
            torch.nn.utils.clip_grad_norm_(inn.parameters(), 1.0)
            optimizer.step()
            
            epoch_nll += nll.item()
            n_batch += 1
        
        scheduler.step()
        avg_nll = epoch_nll / n_batch
        loss_history.append(avg_nll)
        
        # 定期評価
        if (epoch + 1) % 100 == 0:
            inn.eval()
            with torch.no_grad():
                z_all, _ = inn(Z_norm)
            mcc, _ = compute_mcc(z_all.cpu().numpy(), Z_true_np)
            best_mcc = max(best_mcc, mcc)
            if verbose:
                print(f'    Epoch {epoch+1}/{n_epochs}: NLL={avg_nll:.4f}, MCC={mcc:.4f}')
    
    # 最終評価
    inn.eval()
    with torch.no_grad():
        z_all, _ = inn(Z_norm)
    z_hat_np = z_all.cpu().numpy()
    final_mcc, final_perm = compute_mcc(z_hat_np, Z_true_np)
    best_mcc = max(best_mcc, final_mcc)
    
    return z_hat_np, best_mcc, final_mcc, final_perm, loss_history, inn


def run_two_stage(X_np, Z_np, dim, gin_config=None, seed=42, verbose=True):
    """2段階パイプラインの完全実行"""
    if gin_config is None:
        gin_config = {}
    
    defaults = dict(n_coupling=8, subnet_hidden=max(dim, 10),
                    n_epochs=500, lr=5e-4, weight_decay=0.01, batch_size=256)
    defaults.update(gin_config)
    
    t0 = time.time()
    
    # Stage 1: ICA
    if verbose: print(f'  Stage 1: FastICA (dim={dim})...')
    Z_ica, ica_model = stage1_ica(X_np, dim)
    mcc_ica, _ = compute_mcc(Z_ica, Z_np)
    if verbose: print(f'    ICA MCC = {mcc_ica:.4f}')
    
    # Stage 2: GIN
    if verbose: print(f'  Stage 2: GIN (coupling={defaults["n_coupling"]}, h={defaults["subnet_hidden"]})...')
    z_hat_np, best_mcc, final_mcc, perm, loss_history, inn = stage2_gin(
        Z_ica, Z_np, dim, seed=seed, verbose=verbose, **defaults
    )
    
    elapsed = time.time() - t0
    if verbose:
        print(f'  Result: ICA={mcc_ica:.4f} → 2-stage={best_mcc:.4f} ({elapsed:.1f}s)')
    
    return {
        'mcc_ica': mcc_ica,
        'mcc_best': best_mcc,
        'mcc_final': final_mcc,
        'perm': perm.tolist(),
        'z_hat': z_hat_np,
        'loss_history': loss_history,
        'elapsed': elapsed,
    }


print('Pipeline loaded.')

In [ ]:
# === Cell 4: dim=16 確認（ローカル検証済みの再現） ===

print('=' * 60)
print('dim=16 確認テスト（ローカルで MCC=0.754 を達成済み）')
print('=' * 60)

dim = 16
X_np, Z_np, groups = make_nonlinear_data(dim, n_samples=10000)
print(f'Data: dim={dim}, n=10000, nonlinear mixing')

# 最良設定: nc=8, h=16, wd=0.01
result_16 = run_two_stage(X_np, Z_np, dim, gin_config=dict(
    n_coupling=8, subnet_hidden=16, n_epochs=500, weight_decay=0.01
))

print(f'\nICA only:  MCC = {result_16["mcc_ica"]:.4f}')
print(f'ICA→GIN:   MCC = {result_16["mcc_best"]:.4f}')
print(f'Target:    MCC > 0.75')
print(f'Status:    {"PASS" if result_16["mcc_best"] > 0.70 else "FAIL"}')

In [ ]:
# === Cell 5: 高次元テスト（GPU推奨） ===

print('=' * 60)
print('高次元スケーリングテスト（非線形混合）')
print('=' * 60)

results_all = {}

# 各次元の設定
# 高次元ではcoupling層を増やし、subnet容量も適度に増やす
dim_configs = {
    16:  dict(n_coupling=8,  subnet_hidden=16,  n_epochs=500,  n_samples=10000),
    32:  dict(n_coupling=12, subnet_hidden=32,  n_epochs=500,  n_samples=10000),
    64:  dict(n_coupling=16, subnet_hidden=48,  n_epochs=500,  n_samples=10000),
    128: dict(n_coupling=24, subnet_hidden=64,  n_epochs=800,  n_samples=10000),
    256: dict(n_coupling=32, subnet_hidden=96,  n_epochs=800,  n_samples=10000),
}

for dim, cfg in dim_configs.items():
    print(f'\n--- dim={dim} ---')
    n_samples = cfg.pop('n_samples')
    X_np, Z_np, groups = make_nonlinear_data(dim, n_samples=n_samples)
    
    result = run_two_stage(X_np, Z_np, dim, gin_config=cfg)
    results_all[dim] = {
        'mcc_ica': result['mcc_ica'],
        'mcc_best': result['mcc_best'],
        'elapsed': result['elapsed'],
        'config': cfg,
    }
    cfg['n_samples'] = n_samples  # restore

print('\n' + '=' * 60)
print('Summary')
print('=' * 60)
print(f'{"dim":>6} {"ICA only":>10} {"ICA→GIN":>10} {"Δ":>8} {"Time":>8}')
print('-' * 46)
for dim, r in sorted(results_all.items()):
    delta = r['mcc_best'] - r['mcc_ica']
    marker = ' ***' if r['mcc_best'] > 0.75 else ' **' if r['mcc_best'] > 0.60 else ''
    print(f'{dim:>6} {r["mcc_ica"]:>10.4f} {r["mcc_best"]:>10.4f} {delta:>+8.4f} {r["elapsed"]:>7.1f}s{marker}')

In [ ]:
# === Cell 6: 線形混合テスト（比較用） ===

print('=' * 60)
print('線形混合テスト（ICA単体で十分なはず）')
print('=' * 60)

results_linear = {}
for dim in [16, 32, 64, 128, 256]:
    X_np, Z_np, groups = make_linear_data(dim, n_samples=10000)
    Z_ica, _ = stage1_ica(X_np, dim)
    mcc_ica, _ = compute_mcc(Z_ica, Z_np)
    results_linear[dim] = mcc_ica
    print(f'  dim={dim:>4}: ICA MCC = {mcc_ica:.4f}')

print('\n(線形混合ではICA単体で十分。GINは不要。)')

In [ ]:
# === Cell 7: Multi-seed安定性テスト（best dim） ===

# Cell 5 の結果から最もMCCが高かったdimを選択
best_dim = max(results_all, key=lambda d: results_all[d]['mcc_best'])
print(f'Best dim from Cell 5: dim={best_dim} (MCC={results_all[best_dim]["mcc_best"]:.4f})')
print()

# 10 seeds で安定性を確認
print(f'=== Multi-seed test: dim={best_dim}, 10 seeds ===')
cfg = dim_configs[best_dim].copy()
n_samples = cfg.pop('n_samples', 10000)
X_np, Z_np, groups = make_nonlinear_data(best_dim, n_samples=n_samples)

seed_mccs = []
for s in range(10):
    result = run_two_stage(X_np, Z_np, best_dim, gin_config=cfg, seed=s, verbose=False)
    seed_mccs.append(result['mcc_best'])
    print(f'  seed={s}: MCC = {result["mcc_best"]:.4f}')

print(f'\n  Mean = {np.mean(seed_mccs):.4f}')
print(f'  Std  = {np.std(seed_mccs):.4f}')
print(f'  Max  = {max(seed_mccs):.4f}')
print(f'  Min  = {min(seed_mccs):.4f}')
print(f'  >0.75: {sum(1 for m in seed_mccs if m > 0.75)}/10')

In [ ]:
# === Cell 8: 結果の保存 ===

import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# --- JSON保存 ---
save_data = {
    'method': 'Two-stage: FastICA → GIN (NLL, Laplace prior)',
    'nonlinear_mixing': {str(d): {'mcc_ica': r['mcc_ica'], 'mcc_2stage': r['mcc_best'],
                                   'elapsed_s': r['elapsed']}
                         for d, r in results_all.items()},
    'linear_mixing': {str(d): mcc for d, mcc in results_linear.items()},
    'multi_seed': {'dim': best_dim, 'mccs': seed_mccs,
                   'mean': float(np.mean(seed_mccs)), 'std': float(np.std(seed_mccs))},
}

os.makedirs('results', exist_ok=True)
with open('results/01v5_gin_synthetic_results.json', 'w') as f:
    json.dump(save_data, f, indent=2)
print('Saved: results/01v5_gin_synthetic_results.json')

# --- 図1: dim vs MCC ---
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
dims_nl = sorted(results_all.keys())
mccs_ica = [results_all[d]['mcc_ica'] for d in dims_nl]
mccs_2s = [results_all[d]['mcc_best'] for d in dims_nl]
dims_lin = sorted(results_linear.keys())
mccs_lin = [results_linear[d] for d in dims_lin]

ax.plot(dims_nl, mccs_ica, 'o-', label='ICA only (nonlinear mix)', color='tab:orange')
ax.plot(dims_nl, mccs_2s, 's-', label='ICA→GIN (nonlinear mix)', color='tab:blue', linewidth=2)
ax.plot(dims_lin, mccs_lin, '^--', label='ICA only (linear mix)', color='tab:green', alpha=0.7)
ax.axhline(y=0.75, color='red', linestyle=':', label='MCC=0.75 threshold')
ax.set_xlabel('Dimension')
ax.set_ylabel('MCC')
ax.set_title('Two-Stage ICA→GIN: Dimension Scaling')
ax.legend()
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/v5_dim_scaling.png', dpi=150)
print('Saved: results/v5_dim_scaling.png')
plt.show()

# --- 図2: Multi-seed分布 ---
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.bar(range(len(seed_mccs)), seed_mccs, color='tab:blue', alpha=0.7)
ax.axhline(y=0.75, color='red', linestyle=':', label='MCC=0.75')
ax.axhline(y=np.mean(seed_mccs), color='blue', linestyle='--', label=f'Mean={np.mean(seed_mccs):.3f}')
ax.set_xlabel('Seed')
ax.set_ylabel('MCC')
ax.set_title(f'Multi-Seed Stability (dim={best_dim})')
ax.legend()
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig('results/v5_multi_seed.png', dpi=150)
print('Saved: results/v5_multi_seed.png')
plt.show()

print('\nAll results saved.')

In [ ]:
# === Cell 9: GitHub Push（Colab Secrets使用） ===

if IN_COLAB:
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        !git config user.email 'colab@example.com'
        !git config user.name 'Colab Runner'
        !git add results/01v5_gin_synthetic_results.json results/v5_*.png
        !git commit -m 'Add v5 synthetic GIN results'
        !git remote set-url origin https://{token}@github.com/AUMEZAK/thoughtcomm.git
        !git push
        print('Results pushed to GitHub.')
    except Exception as e:
        print(f'Push failed: {e}')
        print('Results saved locally. Manual push needed.')
else:
    print('Not in Colab. Push manually with: git add results/ && git commit && git push')